# Experiment 05 — Asking what an embedding knows

**Question:** How easily can a simple classifier read class information from each frozen representation?

Experiment 04 used nearest neighbours as an early semantic check. Here we train a **linear probe** on each representation. The encoders are frozen, so only the small classifier can learn from the labels.

## Our prediction

If an encoder has already organised images by useful object information, a single linear layer should recover CIFAR-10 classes more accurately than chance. Experiment 04 did not establish a winner, so our comparison is deliberately open: we will let held-out accuracy decide which representation makes class information easier to read in this run.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the JEPA Experiments project root')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset, TensorDataset

from jepa.data import load_cifar10
from jepa.device import best_device
from jepa.masking import centered_block_mask
from jepa.models import PatchAutoencoder, TinyJEPA, masked_embedding_loss, masked_pixel_loss

torch.manual_seed(7)
device = best_device()
device

## 1. Fix the experiment

We repeat the self-supervised training conditions from Experiment 04. The probe receives 10,000 labelled training examples and is evaluated on 2,000 held-out test examples. The encoder never receives a gradient from those labels.

The test set is used once for the final measurement. We do not choose settings by looking at its accuracy.

In [ ]:
PRETRAIN_IMAGES = 10_000
PROBE_TRAIN_IMAGES = 10_000
TEST_IMAGES = 2_000
PRETRAIN_BATCH_SIZE = 64
PROBE_BATCH_SIZE = 256
PRETRAIN_EPOCHS = 5
PROBE_EPOCHS = 30
LEARNING_RATE = 3e-4
PROBE_LEARNING_RATE = 3e-3
EMBEDDING_DIM = 128
PROBE_SEEDS = [11, 22, 33, 44, 55]

mask = centered_block_mask(8, 4, device=device)
full_train_data = load_cifar10(train=True)
full_test_data = load_cifar10(train=False)
pretrain_data = Subset(full_train_data, range(PRETRAIN_IMAGES))
probe_train_data = Subset(full_train_data, range(PROBE_TRAIN_IMAGES))
test_data = Subset(full_test_data, range(TEST_IMAGES))

pretrain_loader = DataLoader(pretrain_data, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
probe_source_loader = DataLoader(probe_train_data, batch_size=256, shuffle=False, num_workers=0)
test_source_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0)

print('Self-supervised training images:', len(pretrain_data))
print('Labelled probe training images:', len(probe_train_data))
print('Held-out test images:', len(test_data))
print('Hidden patches during pretraining:', int(mask.sum()))

## 2. Recreate the two representations

This notebook is standalone, so it trains both models again rather than depending on Python objects left in Experiment 04. Labels are ignored throughout this stage.

The models solve different prediction tasks. Their raw losses still have different units and should not be compared directly.

In [ ]:
pixel_model = PatchAutoencoder(embedding_dim=EMBEDDING_DIM).to(device)
jepa_model = TinyJEPA(embedding_dim=EMBEDDING_DIM, ema_decay=0.99).to(device)

pixel_optimizer = AdamW(pixel_model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
jepa_optimizer = AdamW(
    list(jepa_model.context_encoder.parameters()) + list(jepa_model.predictor.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.05,
)

pixel_losses = []
jepa_losses = []
for epoch in range(PRETRAIN_EPOCHS):
    pixel_model.train()
    jepa_model.train()
    pixel_total = 0.0
    jepa_total = 0.0
    seen = 0

    for images, _ in pretrain_loader:
        images = images.to(device)

        pixel_prediction, pixel_target = pixel_model(images, mask)
        pixel_loss = masked_pixel_loss(pixel_prediction, pixel_target, mask)
        pixel_optimizer.zero_grad(set_to_none=True)
        pixel_loss.backward()
        pixel_optimizer.step()

        jepa_prediction, jepa_target = jepa_model.predict(images, mask)
        jepa_loss = masked_embedding_loss(jepa_prediction, jepa_target, mask)
        jepa_optimizer.zero_grad(set_to_none=True)
        jepa_loss.backward()
        jepa_optimizer.step()
        jepa_model.update_target()

        count = images.shape[0]
        pixel_total += pixel_loss.item() * count
        jepa_total += jepa_loss.item() * count
        seen += count

    pixel_losses.append(pixel_total / seen)
    jepa_losses.append(jepa_total / seen)
    print(
        f'Epoch {epoch + 1:02d}/{PRETRAIN_EPOCHS} | '
        f'pixel MSE={pixel_losses[-1]:.6f} | JEPA loss={jepa_losses[-1]:.6f}'
    )

## 3. Freeze the encoders

Freezing is the defining rule of a linear probe. We disable gradients, put the models in evaluation mode, and confirm that no encoder parameter can change.

For JEPA we use the EMA target encoder, because it supplied the representation targets during self-supervised learning. For the pixel model we use its encoder output before the RGB prediction head.

In [ ]:
pixel_model.eval()
jepa_model.eval()
pixel_model.requires_grad_(False)
jepa_model.requires_grad_(False)

pixel_trainable = sum(p.numel() for p in pixel_model.parameters() if p.requires_grad)
jepa_trainable = sum(p.numel() for p in jepa_model.parameters() if p.requires_grad)
print('Trainable pixel-model parameters:', pixel_trainable)
print('Trainable JEPA parameters:', jepa_trainable)
assert pixel_trainable == 0 and jepa_trainable == 0

## 4. Turn each image into one frozen vector

Each encoder returns 64 patch vectors. We average them to obtain one 128-number vector for the complete image. We also keep flattened RGB pixels as a reference.

We calculate the feature mean and standard deviation from the probe training set only. Applying those same statistics to train and test vectors puts differently scaled features on comparable footing without using test information.

In [ ]:
@torch.inference_mode()
def extract_vectors(loader):
    pixel_vectors = []
    jepa_vectors = []
    raw_vectors = []
    labels = []
    for images, batch_labels in loader:
        images_on_device = images.to(device)
        pixel_vectors.append(pixel_model.encode(images_on_device).mean(dim=1).cpu())
        jepa_vectors.append(jepa_model.target_encoder(images_on_device).mean(dim=1).cpu())
        raw_vectors.append(images.flatten(1))
        labels.append(batch_labels)
    return (
        torch.cat(pixel_vectors),
        torch.cat(jepa_vectors),
        torch.cat(raw_vectors),
        torch.cat(labels),
    )

pixel_train, jepa_train, raw_train, train_labels = extract_vectors(probe_source_loader)
pixel_test, jepa_test, raw_test, test_labels = extract_vectors(test_source_loader)


def standardize(train_vectors, test_vectors):
    mean = train_vectors.mean(dim=0, keepdim=True)
    std = train_vectors.std(dim=0, keepdim=True).clamp_min(1e-6)
    return (train_vectors - mean) / std, (test_vectors - mean) / std

pixel_train, pixel_test = standardize(pixel_train, pixel_test)
jepa_train, jepa_test = standardize(jepa_train, jepa_test)
raw_train, raw_test = standardize(raw_train, raw_test)

print('Pixel representation:', tuple(pixel_train.shape))
print('JEPA representation: ', tuple(jepa_train.shape))
print('Raw-pixel reference: ', tuple(raw_train.shape))

## 5. Train only a linear layer

A probe is `y = Wx + b`: one matrix multiplication and one bias. There are no hidden layers. We repeat it with five seeds because its initial weights and batch order are random.

Accuracy on the training vectors tells us whether the linear layer can fit the available representation. Accuracy on unseen test vectors tells us whether that readable structure generalises.

In [ ]:
def run_linear_probe(train_vectors, train_labels, test_vectors, test_labels, seed):
    torch.manual_seed(seed)
    classifier = nn.Linear(train_vectors.shape[1], 10).to(device)
    optimizer = AdamW(classifier.parameters(), lr=PROBE_LEARNING_RATE, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(train_vectors, train_labels),
        batch_size=PROBE_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    history = []
    for _ in range(PROBE_EPOCHS):
        classifier.train()
        for vectors, labels in loader:
            logits = classifier(vectors.to(device))
            loss = F.cross_entropy(logits, labels.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        classifier.eval()
        with torch.no_grad():
            test_logits = classifier(test_vectors.to(device))
            test_accuracy = (test_logits.argmax(dim=1).cpu() == test_labels).float().mean().item()
        history.append(test_accuracy)

    classifier.eval()
    with torch.no_grad():
        train_predictions = classifier(train_vectors.to(device)).argmax(dim=1).cpu()
        test_predictions = classifier(test_vectors.to(device)).argmax(dim=1).cpu()
    return {
        'train_accuracy': (train_predictions == train_labels).float().mean().item(),
        'test_accuracy': (test_predictions == test_labels).float().mean().item(),
        'history': history,
    }

In [ ]:
representations = {
    'Raw pixels': (raw_train, raw_test),
    'Pixel model': (pixel_train, pixel_test),
    'JEPA': (jepa_train, jepa_test),
}
probe_results = {}

for name, (train_vectors, test_vectors) in representations.items():
    runs = []
    for seed in PROBE_SEEDS:
        result = run_linear_probe(train_vectors, train_labels, test_vectors, test_labels, seed)
        runs.append(result)
        print(
            f'{name:12s} seed={seed:02d} | '
            f'train={result["train_accuracy"]:.1%} | test={result["test_accuracy"]:.1%}'
        )
    probe_results[name] = runs

## 6. Compare held-out accuracy

Chance accuracy is about 10 percent. The most useful comparison is the mean held-out accuracy across seeds. The standard deviation shows how sensitive the answer was to the probe’s random initialization and batch order.

In [ ]:
summary = {}
for name, runs in probe_results.items():
    train_scores = torch.tensor([run['train_accuracy'] for run in runs])
    test_scores = torch.tensor([run['test_accuracy'] for run in runs])
    summary[name] = {
        'train_mean': train_scores.mean().item(),
        'test_mean': test_scores.mean().item(),
        'test_std': test_scores.std().item(),
    }

print('Representation'.ljust(16), 'Train'.rjust(9), 'Test mean'.rjust(12), 'Test std'.rjust(11))
for name, values in summary.items():
    print(
        name.ljust(16),
        f'{values["train_mean"]:.1%}'.rjust(9),
        f'{values["test_mean"]:.1%}'.rjust(12),
        f'{values["test_std"]:.2%}'.rjust(11),
    )

names = list(summary)
means = [summary[name]['test_mean'] * 100 for name in names]
errors = [summary[name]['test_std'] * 100 for name in names]
colors = ['#98A2B3', '#4E7AD9', '#7856C8']

plt.figure(figsize=(8, 4.8))
plt.bar(names, means, yerr=errors, capsize=5, color=colors)
plt.axhline(10, color='#D35B66', linestyle='--', label='Chance (10%)')
plt.ylabel('Held-out accuracy (%)')
plt.title('Linear probe accuracy on frozen representations')
plt.legend()
plt.grid(axis='y', alpha=0.2)
plt.show()

## 7. Inspect learning curves

If accuracy is still rising sharply at epoch 30, the fixed probe budget may be too short. If training accuracy is high while test accuracy is much lower, the probe is fitting details that do not generalise.

In [ ]:
plt.figure(figsize=(8, 4.8))
for name, runs in probe_results.items():
    histories = torch.tensor([run['history'] for run in runs]) * 100
    mean = histories.mean(dim=0)
    std = histories.std(dim=0)
    epochs = torch.arange(1, PROBE_EPOCHS + 1)
    plt.plot(epochs, mean, label=name)
    plt.fill_between(epochs, mean - std, mean + std, alpha=0.15)
plt.axhline(10, color='#D35B66', linestyle='--', label='Chance')
plt.xlabel('Probe epoch')
plt.ylabel('Held-out accuracy (%)')
plt.title('What the linear layer learns while the encoders stay frozen')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

## What happened in this run

Run every cell, then replace this section with the measured results.

Record:

- the mean and standard deviation for the raw-pixel probe;
- the mean and standard deviation for the pixel-model probe;
- the mean and standard deviation for the JEPA probe;
- whether the learning curves had levelled off;
- whether training and test accuracy were far apart.

Then answer the experiment’s question in plain language: **which frozen representation made CIFAR-10 class information easiest for a linear classifier to read?**

Keep the conclusion narrow. This experiment measures linear accessibility under one pretraining run, one pooling method, and one probe recipe. It does not establish that one learning objective is universally better.